# Notebook 08 – Deployment & Documentation

## Objective

Generate deployment-ready documentation for the PharmaOps AI analytics pipeline.

This notebook validates the complete project, documents all datasets, prepares Power BI mappings, creates metadata, and produces final project documentation.

### Deliverables

- Data Dictionary
- Power BI Mapping
- Dataset Lineage
- QA Report
- Project Metadata
- Python Completion Summary

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

import warnings
import os

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns",None)

pd.set_option("display.max_rows",100)

pd.set_option("display.float_format","{:,.2f}".format)

print("="*80)
print("ALL REQUIRED LIBRARIES LOADED")
print("="*80)

ALL REQUIRED LIBRARIES LOADED


In [2]:
documentation_folder = Path(

    "../../outputs/documentation"

)

documentation_folder.mkdir(

    parents=True,

    exist_ok=True

)

print("="*80)
print("DOCUMENTATION FOLDER CREATED")
print("="*80)

print(documentation_folder.resolve())

DOCUMENTATION FOLDER CREATED
C:\Users\venka\OneDrive\Desktop\PharmaOps-AI\outputs\documentation


In [3]:
print("="*80)
print("LOADING PROJECT DATASETS")
print("="*80)

feature_store = pd.read_csv(

    "../../outputs/processed_data/Feature_Store.csv"

)

fact = pd.read_csv(

    "../../outputs/dashboard_data/Fact_Medicine_Performance.csv"

)

dim_medicine = pd.read_csv(

    "../../outputs/dashboard_data/Dim_Medicine.csv"

)

dim_category = pd.read_csv(

    "../../outputs/dashboard_data/Dim_Category.csv"

)

dim_supplier = pd.read_csv(

    "../../outputs/dashboard_data/Dim_Supplier.csv"

)

dim_date = pd.read_csv(

    "../../outputs/dashboard_data/Dim_Date.csv"

)

print("✓ Feature Store")

print("✓ Fact Table")

print("✓ Medicine Dimension")

print("✓ Category Dimension")

print("✓ Supplier Dimension")

print("✓ Date Dimension")

print("="*80)

LOADING PROJECT DATASETS
✓ Feature Store
✓ Fact Table
✓ Medicine Dimension
✓ Category Dimension
✓ Supplier Dimension
✓ Date Dimension


In [4]:
validation = pd.DataFrame({

"Dataset":[

"Feature Store",

"Fact Table",

"Dim Medicine",

"Dim Category",

"Dim Supplier",

"Dim Date"

],

"Rows":[

len(feature_store),

len(fact),

len(dim_medicine),

len(dim_category),

len(dim_supplier),

len(dim_date)

],

"Columns":[

feature_store.shape[1],

fact.shape[1],

dim_medicine.shape[1],

dim_category.shape[1],

dim_supplier.shape[1],

dim_date.shape[1]

],

"Missing Values":[

feature_store.isnull().sum().sum(),

fact.isnull().sum().sum(),

dim_medicine.isnull().sum().sum(),

dim_category.isnull().sum().sum(),

dim_supplier.isnull().sum().sum(),

dim_date.isnull().sum().sum()

]

})

display(validation)

print("="*80)
print("PIPELINE VALIDATION COMPLETED")
print("="*80)


,Dataset,Rows,Columns,Missing Values
0,Feature Store,4954,86,0
1,Fact Table,4954,47,0
2,Dim Medicine,4824,12,0
3,Dim Category,14,3,0
4,Dim Supplier,75,10,0
5,Dim Date,2030,9,0


PIPELINE VALIDATION COMPLETED


# SECTION A – Data Dictionary Generator

## Objective

Automatically generate a complete data dictionary for every dataset produced during the analytics pipeline.

The data dictionary provides column names, data types, null counts, uniqueness, and sample values, enabling easy understanding and maintenance of the datasets.

In [6]:
def create_data_dictionary(df,dataset_name):

    dictionary = pd.DataFrame({

        "Dataset":dataset_name,

        "Column":df.columns,

        "Data_Type":df.dtypes.astype(str).values,

        "Missing_Values":df.isnull().sum().values,

        "Unique_Values":df.nunique().values,

        "Sample_Value":[

            str(df[col].dropna().iloc[0])

            if not df[col].dropna().empty

            else ""

            for col in df.columns

        ]

    })

    return dictionary

In [7]:
print("="*80)
print("GENERATING DATA DICTIONARY")
print("="*80)

data_dictionary = pd.concat([

create_data_dictionary(feature_store,"Feature Store"),

create_data_dictionary(fact,"Fact_Medicine_Performance"),

create_data_dictionary(dim_medicine,"Dim_Medicine"),

create_data_dictionary(dim_category,"Dim_Category"),

create_data_dictionary(dim_supplier,"Dim_Supplier"),

create_data_dictionary(dim_date,"Dim_Date")

],ignore_index=True)

display(

data_dictionary.head(20)

)

GENERATING DATA DICTIONARY


,Dataset,Column,Data_Type,Missing_Values,Unique_Values,Sample_Value
0,Feature Store,Inventory_ID,object,0,4954,INV002361
1,Feature Store,Medicine_ID,object,0,4824,MED000911
2,Feature Store,Supplier_ID,object,0,75,SUP0048
3,Feature Store,Batch_Number,object,0,4954,BAT820020
4,Feature Store,Manufacturing_Date,object,0,1084,2026-03-06
5,Feature Store,Expiry_Date,object,0,1531,2027-03-31
6,Feature Store,Quantity_In_Stock,int64,0,989,993
7,Feature Store,Unit_Cost,float64,0,4753,596.32
8,Feature Store,Selling_Price,float64,0,4788,798.36
9,Feature Store,Reorder_Level,int64,0,176,87


In [8]:
data_dictionary.to_csv(

documentation_folder/

"Data_Dictionary.csv",

index=False

)

print("="*80)
print("DATA DICTIONARY GENERATED SUCCESSFULLY")
print("="*80)

DATA DICTIONARY GENERATED SUCCESSFULLY


# SECTION B – Power BI Mapping Generator

## Objective

Generate a Power BI Mapping document that defines how each dataset should be used within the Power BI semantic model.

The mapping includes:

- Dataset Name
- Table Type
- Primary Key
- Relationships
- Business Purpose
- Dashboard Usage

This document serves as technical documentation for dashboard development.

In [9]:
print("="*80)
print("GENERATING POWER BI MAPPING")
print("="*80)

powerbi_mapping = pd.DataFrame({

    "Table":[

        "Fact_Medicine_Performance",

        "Dim_Medicine",

        "Dim_Category",

        "Dim_Supplier",

        "Dim_Date"

    ],

    "Table_Type":[

        "Fact",

        "Dimension",

        "Dimension",

        "Dimension",

        "Dimension"

    ],

    "Primary_Key":[

        "Inventory_ID",

        "Medicine_ID",

        "Category_ID",

        "Supplier_ID",

        "Date"

    ],

    "Relationship":[

        "Links to all Dimensions",

        "Medicine_ID",

        "Category_ID",

        "Supplier_ID",

        "Date"

    ],

    "Business_Purpose":[

        "Central Business Metrics",

        "Medicine Master",

        "Medicine Classification",

        "Supplier Analytics",

        "Time Intelligence"

    ],

    "Dashboard_Usage":[

        "All Dashboard Pages",

        "Medicine Analysis",

        "Category Analysis",

        "Supplier Dashboard",

        "Time Slicers"

    ]

})

display(powerbi_mapping)

GENERATING POWER BI MAPPING


,Table,Table_Type,Primary_Key,Relationship,Business_Purpose,Dashboard_Usage
0,Fact_Medicine_Performance,Fact,Inventory_ID,Links to all Dimensions,Central Business Metrics,All Dashboard Pages
1,Dim_Medicine,Dimension,Medicine_ID,Medicine_ID,Medicine Master,Medicine Analysis
2,Dim_Category,Dimension,Category_ID,Category_ID,Medicine Classification,Category Analysis
3,Dim_Supplier,Dimension,Supplier_ID,Supplier_ID,Supplier Analytics,Supplier Dashboard
4,Dim_Date,Dimension,Date,Date,Time Intelligence,Time Slicers


In [10]:
powerbi_mapping.to_csv(

    documentation_folder /

    "PowerBI_Mapping.csv",

    index=False

)

print("="*80)
print("POWER BI MAPPING GENERATED")
print("="*80)

POWER BI MAPPING GENERATED


# SECTION C – Dataset Lineage Generator

## Objective

Document the complete flow of data across the PharmaOps AI analytics pipeline.

This provides traceability from raw data to final dashboard datasets.

In [11]:
print("="*80)
print("GENERATING DATASET LINEAGE")
print("="*80)

lineage = pd.DataFrame({

    "Stage":[

        "Raw FDA Dataset",

        "Preprocessing",

        "Feature Engineering",

        "Business Analytics",

        "Dashboard Preparation",

        "AI Recommendation Engine",

        "Power BI Dashboard"

    ],

    "Input":[

        "FDA NDC JSON",

        "Raw Dataset",

        "Processed Dataset",

        "Feature Store",

        "Feature Store",

        "Feature Store",

        "Dashboard Data"

    ],

    "Output":[

        "Medicines_Master",

        "Clean Dataset",

        "Feature_Store.csv",

        "Business Insights",

        "Star Schema",

        "AI Reports",

        "Interactive Dashboard"

    ],

    "Notebook":[

        "Data Engineering",

        "Notebook 02",

        "Notebook 04",

        "Notebook 05",

        "Notebook 06",

        "Notebook 07",

        "Power BI"

    ]

})

display(lineage)

GENERATING DATASET LINEAGE


,Stage,Input,Output,Notebook
0,Raw FDA Dataset,FDA NDC JSON,Medicines_Master,Data Engineering
1,Preprocessing,Raw Dataset,Clean Dataset,Notebook 02
2,Feature Engineering,Processed Dataset,Feature_Store.csv,Notebook 04
3,Business Analytics,Feature Store,Business Insights,Notebook 05
4,Dashboard Preparation,Feature Store,Star Schema,Notebook 06
5,AI Recommendation Engine,Feature Store,AI Reports,Notebook 07
6,Power BI Dashboard,Dashboard Data,Interactive Dashboard,Power BI


In [12]:
lineage.to_csv(

    documentation_folder /

    "Dataset_Lineage.csv",

    index=False

)

print("="*80)
print("DATASET LINEAGE GENERATED")
print("="*80)

DATASET LINEAGE GENERATED


# SECTION D – Star Schema Relationship Matrix

## Objective

Document all relationships between the fact table and dimension tables.

This document simplifies Power BI model development and project maintenance.

In [13]:
relationship_matrix = pd.DataFrame({

    "From_Table":[

        "Fact_Medicine_Performance",

        "Fact_Medicine_Performance",

        "Fact_Medicine_Performance",

        "Fact_Medicine_Performance"

    ],

    "To_Table":[

        "Dim_Medicine",

        "Dim_Category",

        "Dim_Supplier",

        "Dim_Date"

    ],

    "Relationship":[

        "Medicine_ID",

        "Category_ID",

        "Supplier_ID",

        "Date"

    ],

    "Cardinality":[

        "Many-to-One",

        "Many-to-One",

        "Many-to-One",

        "Many-to-One"

    ]

})

display(relationship_matrix)

,From_Table,To_Table,Relationship,Cardinality
0,Fact_Medicine_Performance,Dim_Medicine,Medicine_ID,Many-to-One
1,Fact_Medicine_Performance,Dim_Category,Category_ID,Many-to-One
2,Fact_Medicine_Performance,Dim_Supplier,Supplier_ID,Many-to-One
3,Fact_Medicine_Performance,Dim_Date,Date,Many-to-One


In [14]:
relationship_matrix.to_csv(

    documentation_folder /

    "Relationship_Matrix.csv",

    index=False

)

print("="*80)
print("RELATIONSHIP MATRIX GENERATED")
print("="*80)

RELATIONSHIP MATRIX GENERATED


# SECTION E – Executive Quality Assurance Report

## Objective

Generate a comprehensive quality assurance report for all datasets produced during the analytics pipeline.

The report evaluates:

- Record Count
- Missing Values
- Duplicate Records
- Export Status
- Dataset Quality
- Deployment Readiness

In [15]:
print("="*80)
print("GENERATING QUALITY ASSURANCE REPORT")
print("="*80)

datasets = {

    "Feature_Store":feature_store,

    "Fact_Medicine_Performance":fact,

    "Dim_Medicine":dim_medicine,

    "Dim_Category":dim_category,

    "Dim_Supplier":dim_supplier,

    "Dim_Date":dim_date

}

qa=[]

for name,df in datasets.items():

    completeness = round(
        (1-(df.isnull().sum().sum()/(df.shape[0]*df.shape[1])))*100,2
    )

    duplicate_rows = df.duplicated().sum()

    score = max(

        0,

        100

        - duplicate_rows

        - (100-completeness)

    )

    qa.append({

        "Dataset":name,

        "Rows":len(df),

        "Columns":df.shape[1],

        "Missing_Values":df.isnull().sum().sum(),

        "Duplicate_Rows":duplicate_rows,

        "Completeness(%)":completeness,

        "Quality_Score":round(score,2),

        "Deployment_Status":

            "READY"

            if score>=95

            else "REVIEW"

    })

qa_report=pd.DataFrame(qa)

display(qa_report)

GENERATING QUALITY ASSURANCE REPORT


,Dataset,Rows,Columns,Missing_Values,Duplicate_Rows,Completeness(%),Quality_Score,Deployment_Status
0,Feature_Store,4954,86,0,0,100.00,100.00,READY
1,Fact_Medicine_Performance,4954,47,0,0,100.00,100.00,READY
2,Dim_Medicine,4824,12,0,0,100.00,100.00,READY
3,Dim_Category,14,3,0,0,100.00,100.00,READY
4,Dim_Supplier,75,10,0,0,100.00,100.00,READY
5,Dim_Date,2030,9,0,0,100.00,100.00,READY


In [16]:
print("="*80)
print("PROJECT QUALITY SCORECARD")
print("="*80)

overall_score = round(

    qa_report["Quality_Score"].mean(),

    2

)

deployment = (

    "READY FOR POWER BI"

    if overall_score>=95

    else

    "NEEDS REVIEW"

)

quality_summary = pd.DataFrame({

    "Metric":[

        "Overall Quality Score",

        "Deployment Status",

        "Datasets Validated"

    ],

    "Value":[

        overall_score,

        deployment,

        len(qa_report)

    ]

})

display(quality_summary)

PROJECT QUALITY SCORECARD


,Metric,Value
0,Overall Quality Score,100.00
1,Deployment Status,READY FOR POWER BI
2,Datasets Validated,6


In [17]:
qa_report.to_csv(

documentation_folder/

"QA_Report.csv",

index=False

)

print("="*80)
print("QA REPORT GENERATED")
print("="*80)

QA REPORT GENERATED


# SECTION F – Project Metadata

## Objective

Generate project metadata describing the analytics solution, datasets, technologies, and deployment information.

In [18]:
metadata=f"""

========================================================

PHARMAOPS AI

PROJECT METADATA

========================================================

Project Name

PharmaOps AI

--------------------------------------------------------

Domain

Healthcare Analytics

--------------------------------------------------------

Project Type

End-to-End Data Analytics Platform

--------------------------------------------------------

Technology Stack

Python

Pandas

NumPy

MySQL

Power BI

GitHub

Google Gemini

--------------------------------------------------------

Analytics Pipeline

Notebook 01

Notebook 02

Notebook 03

Notebook 04

Notebook 05

Notebook 06

Notebook 07

Notebook 08

--------------------------------------------------------

Total Datasets

6

--------------------------------------------------------

Total Features

{feature_store.shape[1]}

--------------------------------------------------------

Final Records

{len(feature_store)}

--------------------------------------------------------

Deployment Status

READY

========================================================
"""

print(metadata)




PHARMAOPS AI

PROJECT METADATA


Project Name

PharmaOps AI

--------------------------------------------------------

Domain

Healthcare Analytics

--------------------------------------------------------

Project Type

End-to-End Data Analytics Platform

--------------------------------------------------------

Technology Stack

Python

Pandas

NumPy

MySQL

Power BI

GitHub

Google Gemini

--------------------------------------------------------

Analytics Pipeline

Notebook 01

Notebook 02

Notebook 03

Notebook 04

Notebook 05

Notebook 06

Notebook 07

Notebook 08

--------------------------------------------------------

Total Datasets

6

--------------------------------------------------------

Total Features

86

--------------------------------------------------------

Final Records

4954

--------------------------------------------------------

Deployment Status

READY




In [19]:
with open(

documentation_folder/

"Project_Metadata.txt",

"w",

encoding="utf-8"

) as f:

    f.write(metadata)

print("="*80)
print("PROJECT METADATA GENERATED")
print("="*80)

PROJECT METADATA GENERATED


# SECTION G – Python Completion Report

## Objective

Generate a final project completion report summarizing all work completed during the Python analytics phase.

In [20]:
summary=f"""

========================================================

PYTHON ANALYTICS PHASE COMPLETED

========================================================

Completed

✔ Data Extraction

✔ Data Cleaning

✔ Feature Engineering

✔ Exploratory Data Analysis

✔ Business Analytics

✔ Power BI Star Schema

✔ AI Recommendation Engine

✔ Documentation

✔ QA Validation

========================================================

Outputs Generated

Feature Store

Dashboard Tables

AI Recommendation Reports

Documentation

Metadata

QA Reports

========================================================

STATUS

READY FOR POWER BI DEVELOPMENT

========================================================

"""

print(summary)




PYTHON ANALYTICS PHASE COMPLETED


Completed

✔ Data Extraction

✔ Data Cleaning

✔ Feature Engineering

✔ Exploratory Data Analysis

✔ Business Analytics

✔ Power BI Star Schema

✔ AI Recommendation Engine

✔ Documentation

✔ QA Validation


Outputs Generated

Feature Store

Dashboard Tables

AI Recommendation Reports

Documentation

Metadata

QA Reports


STATUS

READY FOR POWER BI DEVELOPMENT





In [21]:
with open(

documentation_folder/

"Python_Project_Summary.txt",

"w",

encoding="utf-8"

) as f:

    f.write(summary)

print("="*80)
print("PYTHON PROJECT SUMMARY GENERATED")
print("="*80)

PYTHON PROJECT SUMMARY GENERATED


In [22]:
print("="*80)
print("VALIDATING DOCUMENTATION PACKAGE")
print("="*80)

expected_files=[

"Data_Dictionary.csv",

"PowerBI_Mapping.csv",

"Dataset_Lineage.csv",

"Relationship_Matrix.csv",

"QA_Report.csv",

"Project_Metadata.txt",

"Python_Project_Summary.txt"

]

validation=[]

for file in expected_files:

    path=documentation_folder/file

    validation.append({

        "File":file,

        "Exists":path.exists(),

        "Size(KB)":round(path.stat().st_size/1024,2)

        if path.exists()

        else 0

    })

validation=pd.DataFrame(validation)

display(validation)

print()

print("="*80)

print("DOCUMENTATION PACKAGE READY")

print("="*80)

VALIDATING DOCUMENTATION PACKAGE


,File,Exists,Size(KB)
0,Data_Dictionary.csv,True,9.21
1,PowerBI_Mapping.csv,True,0.50
2,Dataset_Lineage.csv,True,0.46
3,Relationship_Matrix.csv,True,0.28
4,QA_Report.csv,True,0.36
5,Project_Metadata.txt,True,1.13
6,Python_Project_Summary.txt,True,0.74



DOCUMENTATION PACKAGE READY


# 🎉 Python Phase Successfully Completed

## Deliverables

### Analytics

- Feature Engineering
- Business Analytics
- AI Recommendation Engine

### Dashboard

- Power BI Star Schema
- Dashboard Datasets

### Documentation

- Data Dictionary
- Power BI Mapping
- Dataset Lineage
- Relationship Matrix
- QA Report
- Metadata
- Python Completion Summary

---

## Final Status

The Python analytics pipeline has been successfully completed.

The project is now fully prepared for Power BI dashboard development.